# 01 — Data Exploration: California Wildfires

**Goal:** Load the wildfire dataset and get to know it *before* cleaning or analyzing anything.

**Dataset:** Fire Program Analysis Fire-Occurrence Database (FPA FOD), 6th Edition, USDA Forest Service.
Short, Karen C. 2022. *Spatial wildfire occurrence data for the United States, 1992-2020* [FPA_FOD_20221014].
https://doi.org/10.2737/RDS-2013-0009.6

The data comes as a SQLite database file (`data/raw/FPA_FOD_20221014.sqlite`). It covers the whole United States,
so we will only load the California rows.

In this notebook we will look at:
1. The first few rows
2. The shape (rows × columns)
3. Column names and data types
4. Missing values
5. Basic statistics
6. Obvious data-quality problems

## 1. Imports

- **pandas** (`pd`) — works with tables of data (called *DataFrames*).
- **NumPy** (`np`) — fast math on numbers; pandas is built on top of it.
- **sqlite3** — built into Python; lets us open the `.sqlite` database file.

In [1]:
import sqlite3

import numpy as np
import pandas as pd

## 2. Load the California wildfire records

A SQLite database can hold several tables. The wildfire records are in a table called `Fires`.

We use a SQL query to ask for only the columns we need and only rows where `STATE = 'CA'`.
We skip the `Shape` column: it stores the map point in a binary format we don't need, because
`LATITUDE` and `LONGITUDE` already give us the location.

The file path starts with `../` because this notebook lives inside the `notebooks/` folder,
so `..` means "go up one folder" to the project root.

In [2]:
DATABASE_PATH = "../data/raw/FPA_FOD_20221014.sqlite"

query = """
SELECT
    FOD_ID,
    FIRE_NAME,
    FIRE_YEAR,
    DISCOVERY_DATE,
    DISCOVERY_DOY,
    CONT_DATE,
    NWCG_CAUSE_CLASSIFICATION,
    NWCG_GENERAL_CAUSE,
    FIRE_SIZE,
    FIRE_SIZE_CLASS,
    LATITUDE,
    LONGITUDE,
    OWNER_DESCR,
    STATE,
    COUNTY,
    FIPS_NAME
FROM Fires
WHERE STATE = 'CA'
"""

connection = sqlite3.connect(DATABASE_PATH)
wildfires = pd.read_sql_query(query, connection)
connection.close()

## 3. First look

`head()` shows the first 5 rows. It's the quickest way to see what the data actually looks like.

In [3]:
wildfires.head()

,FOD_ID,FIRE_NAME,FIRE_YEAR,DISCOVERY_DATE,DISCOVERY_DOY,CONT_DATE,NWCG_CAUSE_CLASSIFICATION,NWCG_GENERAL_CAUSE,FIRE_SIZE,FIRE_SIZE_CLASS,LATITUDE,LONGITUDE,OWNER_DESCR,STATE,COUNTY,FIPS_NAME
0,1,FOUNTAIN,2005,2/2/2005,33,2/2/2005,Human,Power generation/transmission/distribution,0.10,A,40.036944,-121.005833,USFS,CA,63,Plumas County
1,2,PIGEON,2004,5/12/2004,133,5/12/2004,Natural,Natural,0.25,A,38.933056,-120.404444,USFS,CA,61,Placer County
2,3,SLACK,2004,5/31/2004,152,5/31/2004,Human,Debris and open burning,0.10,A,38.984167,-120.735556,STATE OR PRIVATE,CA,17,El Dorado County
3,4,DEER,2004,6/28/2004,180,7/3/2004,Natural,Natural,0.10,A,38.559167,-119.913333,USFS,CA,3,Alpine County
4,5,STEVENOT,2004,6/28/2004,180,7/3/2004,Natural,Natural,0.10,A,38.559167,-119.933056,USFS,CA,3,Alpine County


## 4. Shape

`shape` returns `(number of rows, number of columns)`. Each row is one wildfire.

In [4]:
wildfires.shape

(251881, 16)

## 5. Column names and data types

- `columns` lists every column name.
- `info()` shows each column's data type and how many non-missing values it has.

Pay attention to the data types. For example, is `DISCOVERY_DATE` stored as a real date, or as text?

In [5]:
wildfires.columns

Index(['FOD_ID', 'FIRE_NAME', 'FIRE_YEAR', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'CONT_DATE', 'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE',
       'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE', 'LONGITUDE', 'OWNER_DESCR',
       'STATE', 'COUNTY', 'FIPS_NAME'],
      dtype='object')

In [6]:
wildfires.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251881 entries, 0 to 251880
Data columns (total 16 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   FOD_ID                     251881 non-null  int64  
 1   FIRE_NAME                  204514 non-null  object 
 2   FIRE_YEAR                  251881 non-null  int64  
 3   DISCOVERY_DATE             251881 non-null  object 
 4   DISCOVERY_DOY              251881 non-null  int64  
 5   CONT_DATE                  134002 non-null  object 
 6   NWCG_CAUSE_CLASSIFICATION  251881 non-null  object 
 7   NWCG_GENERAL_CAUSE         251881 non-null  object 
 8   FIRE_SIZE                  251881 non-null  float64
 9   FIRE_SIZE_CLASS            251881 non-null  object 
 10  LATITUDE                   251881 non-null  float64
 11  LONGITUDE                  251881 non-null  float64
 12  OWNER_DESCR                251881 non-null  object 
 13  STATE                      25

## 6. What do the columns mean?

The dataset comes with a file describing every variable. Let's read it instead of guessing.

In [7]:
variable_descriptions = pd.read_csv("../data/raw/_variable_descriptions.csv")
variable_descriptions

,Table,Variable,Description
0,Fires,FOD_ID,Unique numeric record identifier.
1,Fires,FPA_ID,Unique identifier that contains information ne...
2,Fires,SOURCE_SYSTEM_TYPE,Type of source database or system that the rec...
3,Fires,SOURCE_SYSTEM,Name of or other identifier for source databas...
4,Fires,NWCG_REPORTING_AGENCY,Active National Wildlife Coordinating Group (N...
5,Fires,NWCG_REPORTING_UNIT_ID,Active NWCG Unit Identifier for the unit prepa...
6,Fires,NWCG_REPORTING_UNIT_NAME,Active NWCG Unit Name for the unit preparing t...
7,Fires,SOURCE_REPORTING_UNIT,Code for the agency unit preparing the fire re...
8,Fires,SOURCE_REPORTING_UNIT_NAME,Name of reporting agency unit preparing the fi...
9,Fires,LOCAL_FIRE_REPORT_ID,Number or code that uniquely identifies an inc...


## 7. Missing values

`isna()` marks each missing cell as `True`; `.sum()` then counts them per column.
Dividing by the number of rows gives the **percentage** missing, which is easier to judge.

In [8]:
missing_counts = wildfires.isna().sum()
missing_percent = (missing_counts / len(wildfires) * 100).round(1)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent,
})
missing_summary.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
CONT_DATE,117879,46.8
COUNTY,94725,37.6
FIPS_NAME,94725,37.6
FIRE_NAME,47367,18.8
FOD_ID,0,0.0
FIRE_YEAR,0,0.0
DISCOVERY_DATE,0,0.0
DISCOVERY_DOY,0,0.0
NWCG_CAUSE_CLASSIFICATION,0,0.0
NWCG_GENERAL_CAUSE,0,0.0


**Write down:** Which columns have the most missing values? Does it matter for our research question?
(Hint: we can still locate a fire using latitude/longitude even if its county is missing.)

## 8. Descriptive statistics

`describe()` gives count, mean, standard deviation, min, quartiles, and max for numeric columns.

Look closely at `FIRE_SIZE` (acres). Compare the **mean** with the **median** (the `50%` row).
If the mean is much larger than the median, a small number of huge fires are pulling the average up.
That's called a **right-skewed** distribution.

In [9]:
wildfires.describe()

,FOD_ID,FIRE_YEAR,DISCOVERY_DOY,FIRE_SIZE,LATITUDE,LONGITUDE
count,2.518810e+05,251881.000000,251881.000000,251881.000000,251881.000000,251881.000000
mean,1.426294e+08,2006.010457,200.923678,83.082573,37.251235,-120.118132
std,1.748861e+08,8.302032,67.495270,3039.451969,2.559057,2.130694
min,1.000000e+00,1992.000000,1.000000,0.001000,32.537406,-124.402883
25%,3.848370e+05,1999.000000,162.000000,0.100000,34.661935,-121.680684
50%,1.374468e+06,2006.000000,201.000000,0.200000,37.396107,-120.491111
75%,4.000015e+08,2013.000000,246.000000,1.000000,39.226593,-118.345556
max,4.007330e+08,2020.000000,366.000000,589368.000000,42.417500,-114.125800


In [10]:
print("Mean fire size (acres):  ", wildfires["FIRE_SIZE"].mean())
print("Median fire size (acres):", wildfires["FIRE_SIZE"].median())

Mean fire size (acres):   83.08257312778653
Median fire size (acres): 0.2


## 9. Fire size classes

`FIRE_SIZE_CLASS` groups fires by size. According to the NWCG standard:

| Class | Acres |
|---|---|
| A | 0 – 0.25 |
| B | 0.26 – 9.9 |
| C | 10 – 99.9 |
| D | 100 – 299 |
| E | 300 – 999 |
| F | 1,000 – 4,999 |
| G | 5,000+ |

`value_counts()` counts how many rows fall into each category.

In [11]:
wildfires["FIRE_SIZE_CLASS"].value_counts().sort_index()

FIRE_SIZE_CLASS
A    138489
B     96162
C     11802
D      2555
E      1389
F       946
G       538
Name: count, dtype: int64

## 10. Other quick checks

**Years covered** and **fire causes**:

In [12]:
print("Earliest year:", wildfires["FIRE_YEAR"].min())
print("Latest year:  ", wildfires["FIRE_YEAR"].max())

Earliest year: 1992
Latest year:   2020


In [13]:
wildfires["NWCG_GENERAL_CAUSE"].value_counts()

NWCG_GENERAL_CAUSE
Missing data/not specified/undetermined       95263
Equipment and vehicle use                     45660
Natural                                       30290
Arson/incendiarism                            24412
Debris and open burning                       20861
Recreation and ceremony                       11453
Misuse of fire by a minor                      9316
Smoking                                        8472
Power generation/transmission/distribution     3677
Other causes                                    784
Railroad operations and maintenance             773
Fireworks                                       645
Firearms and explosives use                     275
Name: count, dtype: int64

**Latitude/longitude sanity check.** California spans roughly latitude 32.5° to 42° N and
longitude −124.5° to −114° W. Any points far outside that box would be data errors.

In [14]:
outside_california = wildfires[
    (wildfires["LATITUDE"] < 32) | (wildfires["LATITUDE"] > 42.1) |
    (wildfires["LONGITUDE"] < -124.6) | (wildfires["LONGITUDE"] > -114)
]
print("Records outside California's rough bounding box:", len(outside_california))

Records outside California's rough bounding box: 1


**Duplicate check.** `FOD_ID` is the unique ID for each fire record, so it should never repeat.

In [15]:
print("Duplicate FOD_ID values:", wildfires["FOD_ID"].duplicated().sum())
print("Fully duplicated rows:  ", wildfires.duplicated().sum())

Duplicate FOD_ID values: 0


Fully duplicated rows:   0


**How are the dates stored?**

In [16]:
print(wildfires["DISCOVERY_DATE"].head())
print("Data type:", wildfires["DISCOVERY_DATE"].dtype)

0     2/2/2005
1    5/12/2004
2    5/31/2004
3    6/28/2004
4    6/28/2004
Name: DISCOVERY_DATE, dtype: object
Data type: object


## 11. Data-quality notes

Observations from running this notebook (California, 1992–2020):

- **Size of the data:** 251,881 California fire records and 16 selected columns.
- **Missing values:** `CONT_DATE` (containment date) is missing for 46.8% of records, `COUNTY`/`FIPS_NAME` for 37.6%,
  and `FIRE_NAME` for 18.8%. Location is still usable because `LATITUDE` and `LONGITUDE` have no missing values,
  so we will locate fires by coordinates rather than relying on county.
- **Dates:** `DISCOVERY_DATE` is stored as text like `2/2/2005` (month/day/year), not as a real date.
  It must be converted with `pd.to_datetime` before we can extract months or match weather by date.
- **Fire size is extremely skewed:** the mean is about 83 acres but the median is only 0.2 acres, and the
  largest fire is 589,368 acres. A handful of huge fires dominate the average, so we will prefer medians
  and use a log transformation for analysis.
- **Most fires are tiny:** 138,489 are class A (≤ 0.25 acres) and 96,162 are class B (0.26–9.9 acres),
  about 93% of all records. Only 538 are class G (5,000+ acres).
- **Causes:** the largest single category is "Missing data/not specified/undetermined" (95,263 records),
  so cause-based conclusions would be limited.
- **Locations:** only 1 record falls outside a rough California bounding box.
- **Duplicates:** no duplicate `FOD_ID` values and no fully duplicated rows.
